In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import os

import albumentations as A
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from albumentations.pytorch import ToTensorV2
from matplotlib import pyplot as plt
from sklearn.metrics import roc_auc_score
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import transforms

from custom_datasets import CBISDDSM_ResNet_Dataset
from cv_utils import get_cv_folds, train_fold, summarize_cv
from util import (create_dataframes, pair_images, patient_grouped_split, get_model_file_path,
                  run_inference, evaluate_model)

In [ ]:
df, full_mammograms_df, roi_masks_df, cropped_df = create_dataframes()

print(f"Total ROIs: {len(roi_masks_df)}")
print(roi_masks_df['image_path'].head(5))

In [ ]:
paired_df, merged_df = pair_images(roi_masks_df, full_mammograms_df)

In [ ]:
cropped_df = cropped_df[cropped_df['PatientID'].isin(merged_df['PatientID_roi'])]

In [ ]:
train_transform = A.Compose(
    [
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ]
)

test_transform = A.Compose(
    [
        A.Resize(256, 256),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ]
)

inv_transform = transforms.Normalize(
    mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
    std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
)

In [ ]:
dataset = CBISDDSM_ResNet_Dataset(dataframe=cropped_df)

transformed_dataset = CBISDDSM_ResNet_Dataset(dataframe=cropped_df, transform=test_transform)

In [ ]:
dataset.visualize_sample(0, "Original image")

transformed_dataset.visualize_sample(0, "Transformed image")

# Train test split

In [ ]:
train_df, val_df, test_df = patient_grouped_split(cropped_df, val_size=0.15, test_size=0.15, random_state=42)

In [ ]:
train_dataset = CBISDDSM_ResNet_Dataset(dataframe=train_df, transform=train_transform)

val_dataset = CBISDDSM_ResNet_Dataset(dataframe=val_df, transform=test_transform)

test_dataset = CBISDDSM_ResNet_Dataset(dataframe=test_df, transform=test_transform)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=True,
)

# Cross-validate over train + val only. Folding over the whole `cropped_df` would put test
# patients into the training half of four folds out of five, so the test set would no longer
# be held out and its reported numbers would be optimistic.
cv_df = pd.concat([train_df, val_df], ignore_index=True)

folds = get_cv_folds(cv_df, n_splits=5, random_state=42)

# Initialize model and check for cuda

In [ ]:
def get_model(num_classes):
    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    for param in m.parameters():
        param.requires_grad = False

    # for param in m.layer4.parameters():
    #     param.requires_grad = True
    # for param in m.layer3.parameters():
    #     param.requires_grad = True
    # for param in m.layer2.parameters():
    #     param.requires_grad = True

    num_features = m.fc.in_features
    m.fc = nn.Linear(num_features, num_classes)
    return m


num_classes = 2
model = get_model(num_classes)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
print(f"Using device: {device}")

In [ ]:
checkpoint_dir = '../models/Resnet/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

completed_dir = '../models/Resnet/completed'
os.makedirs(completed_dir, exist_ok=True)

# Check for trained models

In [ ]:
completed_path = None

if len(os.listdir(completed_dir)) > 0:
    completed_path = get_model_file_path("Completed model directory is not empty. Do you want to use saved model? (y/n): ",
                                         "../models/Resnet/completed")

    if completed_path:
        print(f"Using saved model from {completed_path}")
    else:
        print("No model file selected. Starting training from scratch.")

# Model training

In [ ]:
class_weights = torch.tensor([1.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

fold_histories = []
fold_models = []

for i, (tr_df, va_df) in enumerate(folds):
    print()
    print("=" * 70)
    print(f"Fold {i + 1}/{len(folds)}")
    print("=" * 70)

    tr_ds = CBISDDSM_ResNet_Dataset(tr_df, train_transform)
    va_ds = CBISDDSM_ResNet_Dataset(va_df, test_transform)

    tr_loader = DataLoader(
        tr_ds,
        batch_size=64,
        shuffle=True,
        pin_memory=True,
    )

    va_loader = DataLoader(
        va_ds,
        batch_size=64,
        shuffle=False,  # Evaluation order must be deterministic; shuffling here buys nothing.
        pin_memory=True,
    )

    # Every fold gets a fresh model, optimizer and scheduler. Carrying them over would mean
    # fold i+1 starts from weights already fitted on its own validation patients, and would
    # also inherit the previous fold's Adam moments and decayed learning rate.
    fold_model = get_model(num_classes).to(device)
    params = [p for p in fold_model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=0.0001)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

    fold_model, history = train_fold(
        fold_model,
        tr_loader,
        va_loader,
        criterion,
        optimizer,
        scheduler,
        num_epochs=20,
        device=device,
        early_stopping_patience=10,
    )

    fold_histories.append(history)
    fold_models.append(fold_model)

cv_summary = summarize_cv(fold_histories)

# Keep the best fold's weights for the downstream test-set evaluation. Because the folds were
# built from train + val only, no fold ever saw a test patient.
best_fold = int(np.argmax([h['best_val_auc'] for h in fold_histories]))
model = fold_models[best_fold]
print()
print(f"Using fold {best_fold + 1} for test evaluation "
      f"(val AUC {fold_histories[best_fold]['best_val_auc']:.4f}).")

In [ ]:
if len(fold_histories) > 0:
    fig, (loss_ax, auc_ax) = plt.subplots(1, 2, figsize=(13, 5))
    colors = plt.get_cmap('tab10').colors

    for f, h in enumerate(fold_histories):
        c = colors[f % len(colors)]
        loss_ax.plot(h['train_loss'], color=c, linestyle='--', alpha=0.6,
                     label=f'Fold {f + 1} train')
        loss_ax.plot(h['val_loss'], color=c, label=f'Fold {f + 1} val')
        loss_ax.plot(h['best_epoch'], h['best_val_loss'], 'o', color=c, markersize=6)

        auc_ax.plot(h['val_auc'], color=c, label=f'Fold {f + 1}')
        auc_ax.plot(h['best_epoch'], h['best_val_auc'], 'o', color=c, markersize=6)

    loss_ax.set_xlabel('Epoch')
    loss_ax.set_ylabel('Loss')
    loss_ax.set_title('Per-fold Loss (markers = selected epoch)')
    loss_ax.legend(fontsize=8, ncol=2)
    loss_ax.grid(alpha=0.3)

    auc_ax.axhline(0.5, color='k', linestyle='--', lw=1, label='Chance (0.5)')
    auc_ax.axhline(cv_summary['val_auc_mean'], color='tab:red', linestyle=':', lw=1.5,
                   label=f"Mean best AUC ({cv_summary['val_auc_mean']:.3f})")
    auc_ax.set_xlabel('Epoch')
    auc_ax.set_ylabel('ROC-AUC')
    auc_ax.set_title('Per-fold Validation ROC-AUC')
    auc_ax.legend(fontsize=8)
    auc_ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No training history to plot.")

In [ ]:
if not completed_path:
    for i, fold_model in enumerate(fold_models):
        fold_save_path = os.path.join(completed_dir, f'fold_{i + 1}.pth')
        torch.save(fold_model.state_dict(), fold_save_path)
    print(f"Saved {len(fold_models)} fold models to {completed_dir}")

    save_path = os.path.join(completed_dir, 'completed.pth')
    torch.save(model.state_dict(), save_path)
    print(f"Best fold (fold {best_fold + 1}) saved to {save_path}")

# Testing and Visualization

## Visualization

In [ ]:
def test_and_visualize(model, loader, device, num_samples=3):
    model.eval()

    images, labels = next(iter(loader))
    images = images.to(device)

    with torch.no_grad():
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

    images = images.cpu()
    labels = labels.cpu()
    preds = preds.cpu()

    class_names = ['benign', 'malignant']

    num_to_plot = min(num_samples, images.size(0))
    fig, axes = plt.subplots(1, num_to_plot, figsize=(15, 4))

    if num_to_plot == 1:
        axes = [axes]

    for i in range(num_to_plot):
        ax = axes[i]
        img = inv_transform(images[i])

        img = img.permute(1, 2, 0).numpy()

        img = np.clip(img, 0, 1)

        ax.imshow(img)

        true_label = class_names[labels[i].item()]
        pred_label = class_names[preds[i].item()]

        color = 'green' if preds[i] == labels[i] else 'red'
        ax.set_title(f"True: {true_label}, Pred: {pred_label}", color=color)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
test_and_visualize(model, test_loader, device)

## Evaluation

In [ ]:
# `val_df` is part of the cross-validation pool, so the selected fold trained on most of it.
# Only the test split is genuinely held out, so that is the number to report.
evaluate_model(model, test_loader, criterion, device, split_name='Test')

In [ ]:
# Ensemble the folds: average P(malignant) across all fold models. This is usually worth a
# point or two of AUC over any single fold and costs nothing but inference time.
fold_probs = []
test_labels = None

for fold_model in fold_models:
    labels, probs, _ = run_inference(fold_model, test_loader, criterion, device)
    fold_probs.append(probs)
    test_labels = labels

ensemble_probs = np.mean(fold_probs, axis=0)
print(f"Ensemble test ROC-AUC: {roc_auc_score(test_labels, ensemble_probs):.4f}")
print(f"Best single fold test ROC-AUC: "
      f"{max(roc_auc_score(test_labels, p) for p in fold_probs):.4f}")